# DriveSense-VLM: Complete Pipeline (Quick Start)

Runs the **entire DriveSense-VLM pipeline** end-to-end in a single notebook:
data pipeline → SFT training → model optimization → benchmarks → 4-level evaluation.

For a detailed, step-by-step walkthrough with explanations, use notebooks `00`–`04` instead.

## ⚠️ Before Running

1. **Set Runtime → Change runtime type → A100 GPU** before starting.
   Training will OOM on GPUs < 40 GB VRAM.

2. **Estimated total time**: 8–10 hours (mostly training).

3. **Estimated cost**: ~110 CU (out of ~200 CU total Colab Pro budget).
   | Stage | CU |
   |-------|----|
   | Data pipeline (mock annotation) | ~5 |
   | SFT training (A100, ~6 h) | ~72 |
   | Optimization (A100, ~1.5 h) | ~18 |
   | Benchmarks (A100, ~1 h) | ~12 |
   | Evaluation (A100, ~1 h) | ~12 |
   | **Total** | **~119** |

4. **Disconnect recovery**: All checkpoints and outputs are saved to Google Drive.
   If Colab disconnects, rerun Cell 3 (setup), then rerun the training cell —
   it auto-resumes from the latest checkpoint. All other stages are idempotent.

5. **nuScenes data**: Download separately and place at
   `DriveSense-VLM/data/nuscenes/` on Drive, **or** use the nuScenes mini
   download cell below for a quick test run.

In [ ]:
# Cell 3: Setup — Drive mount, clone, install ALL deps
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

# Install all dependency groups
!pip install -e '.[data,training,inference,eval]' -q
!pip install pyspark>=3.5 autoawq vllm -q
!pip install flash-attn --no-build-isolation -q
!pip install tensorrt -q 2>/dev/null || echo 'TensorRT unavailable — using torch.compile fallback.'

import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → A100'
print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
print('Setup complete.')

In [ ]:
# Cell 4: Data pipeline
# Uses --mock-llm for annotation (no API key required).
# Replace --mock-llm with your ANTHROPIC_API_KEY for real annotations.
print('=== STAGE 1/5: Data Pipeline ===')

# nuScenes mini download (skip if already on Drive)
from pathlib import Path
nuscenes_mini = Path(PROJECT_ROOT) / 'data' / 'nuscenes' / 'v1.0-mini'
if not nuscenes_mini.exists():
    print('nuScenes mini not found — attempting download ...')
    !pip install nuscenes-devkit -q
    # Place your download command here if you have a pre-signed URL:
    # !wget -q -O /tmp/nuscenes_mini.tar.gz "<URL>" && tar -xzf /tmp/nuscenes_mini.tar.gz -C {PROJECT_ROOT}/data/nuscenes/
    print('Skipping nuScenes download — add your download command above.')

# nuScenes rarity filtering
!python scripts/run_nuscenes_filter.py \
    --nuscenes-root {PROJECT_ROOT}/data/nuscenes \
    --version v1.0-mini \
    --output-dir {PROJECT_ROOT}/outputs/data/nuscenes_filtered 2>/dev/null || echo 'Filter skipped (no data).'

# PySpark pipeline
!python scripts/run_spark_pipeline.py \
    --version v1.0-mini \
    --nuscenes-root {PROJECT_ROOT}/data/nuscenes \
    --output-dir {PROJECT_ROOT}/outputs/data/spark_processed 2>/dev/null || echo 'Spark skipped (no data).'

# Build unified dataset
!python scripts/run_build_unified_dataset.py \
    --nuscenes-dir {PROJECT_ROOT}/outputs/data/nuscenes_filtered \
    --output-dir {PROJECT_ROOT}/outputs/data/unified 2>/dev/null || echo 'Unified build skipped.'

# LLM annotation (mock)
!python scripts/run_annotation_pipeline.py \
    --unified-dir {PROJECT_ROOT}/outputs/data/unified \
    --output-dir {PROJECT_ROOT}/outputs/data/annotated \
    --cache-dir {PROJECT_ROOT}/outputs/data/annotation_cache \
    --mock-llm

# SFT formatting
!python scripts/run_annotation_pipeline.py \
    --format-only \
    --annotated-dir {PROJECT_ROOT}/outputs/data/annotated \
    --output-dir {PROJECT_ROOT}/outputs/data/sft_ready

print('\n=== Data pipeline complete ===')

In [ ]:
# Cell 5: SFT Training
# RECOVERY: If Colab disconnects, rerun Cell 3 (setup) then rerun this cell.
# Training auto-resumes from the latest checkpoint on Google Drive.
print('=== STAGE 2/5: SFT Training ===')

# Copy SFT data to fast ephemeral storage
!mkdir -p /content/sft_data
!cp {PROJECT_ROOT}/outputs/data/sft_ready/sft_train.jsonl /content/sft_data/ 2>/dev/null || true
!cp {PROJECT_ROOT}/outputs/data/sft_ready/sft_val.jsonl /content/sft_data/ 2>/dev/null || true

!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={PROJECT_ROOT}/outputs/training \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume

print('\n=== Training complete ===')

In [ ]:
# Cell 6: Model Optimization (merge → AWQ → TensorRT)
# Each stage is idempotent — safe to rerun after disconnect.
print('=== STAGE 3/5: Optimization ===')

import glob
from pathlib import Path

training_dir = Path(PROJECT_ROOT) / 'outputs' / 'training'
best = training_dir / 'checkpoint-best'
if not best.exists():
    ckpts = sorted(glob.glob(str(training_dir / 'checkpoint-*')))
    best = Path(ckpts[-1]) if ckpts else None

if best:
    !python scripts/run_optimize_model.py --all \
        --adapter-path {best} \
        --override inference.merge.output_dir={PROJECT_ROOT}/outputs/merged_model \
        --override inference.quantization.output_dir={PROJECT_ROOT}/outputs/quantized_model \
        --override inference.tensorrt.output_dir={PROJECT_ROOT}/outputs/tensorrt
else:
    print('No checkpoint found — running optimization in --mock mode for demo.')
    !python scripts/run_optimize_model.py --all --mock \
        --override inference.merge.output_dir={PROJECT_ROOT}/outputs/merged_model \
        --override inference.quantization.output_dir={PROJECT_ROOT}/outputs/quantized_model \
        --override inference.tensorrt.output_dir={PROJECT_ROOT}/outputs/tensorrt

print('\n=== Optimization complete ===')

In [ ]:
# Cell 7: Inference Benchmarks
print('=== STAGE 4/5: Benchmarks ===')

from pathlib import Path
quant_dir = Path(PROJECT_ROOT) / 'outputs' / 'quantized_model'
MOCK_FLAG = '' if quant_dir.exists() else '--mock'

!python scripts/run_benchmark.py \
    --vllm \
    --output {PROJECT_ROOT}/outputs/benchmarks/vllm_bench.json \
    {MOCK_FLAG}

!python scripts/run_benchmark.py \
    --local \
    --output {PROJECT_ROOT}/outputs/benchmarks/local_bench.json \
    {MOCK_FLAG}

!python scripts/run_benchmark.py \
    --vit-only \
    --output {PROJECT_ROOT}/outputs/benchmarks/vit_bench.json \
    {MOCK_FLAG}

print('\n=== Benchmarks complete ===')

In [ ]:
# Cell 8: Full 4-Level Evaluation
print('=== STAGE 5/5: Evaluation ===')

import glob, os
from pathlib import Path

# Check for checkpoint
training_dir = Path(PROJECT_ROOT) / 'outputs' / 'training'
best = training_dir / 'checkpoint-best'
if not best.exists():
    ckpts = sorted(glob.glob(str(training_dir / 'checkpoint-*')))
    best = Path(ckpts[-1]) if ckpts else None

MOCK_FLAG = '--mock' if best is None else ''
adapter_flag = f'--adapter-path {best}' if best else ''

# Generate predictions
!python scripts/run_generate_predictions.py \
    --split test \
    {adapter_flag} \
    --output {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    {MOCK_FLAG}

# Run all 4 evaluation levels
!python scripts/run_full_evaluation.py \
    --level 1 2 3 4 \
    --mock-judge \
    --benchmarks-dir {PROJECT_ROOT}/outputs/benchmarks \
    --predictions {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    --output-dir {PROJECT_ROOT}/outputs/eval \
    {MOCK_FLAG}

print('\n=== Evaluation complete ===')

In [ ]:
# Cell 9: Final report display
from pathlib import Path

# Generate compiled report
!python scripts/run_full_evaluation.py \
    --generate-report \
    --output-dir {PROJECT_ROOT}/outputs/eval

print()
print('=' * 66)
print('  DriveSense-VLM — Pipeline Complete')
print('=' * 66)

# Display report
for candidate in [
    Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'final_report.txt',
    Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'evaluation_report.txt',
]:
    if candidate.exists():
        print(candidate.read_text())
        break

print()
print('All outputs saved to Google Drive:')
print(f'  {PROJECT_ROOT}/outputs/')
print()
print('Next steps:')
print('  1. Review MODEL_CARD.md and fill in actual metric values')
print('  2. Upload model to HuggingFace Hub')
print('  3. Deploy demo/app.py to HF Spaces (T4 GPU)')